# Sieve · run a job on a Colab GPU

1. In the web app, open a step (Screening or Extraction), pick the records, and under **Run on Colab / a local GPU** click **Export job file**.
2. Here: *Runtime → Change runtime type → GPU* (a T4 works with 4-bit; L4/A100 without).
3. Run the cells, upload the job file, wait, download the results file.
4. In the web app: **Import results file**. The run appears as a column next to your other runs.

The job file holds the exact prompts and JSON schema the app would send, so results are comparable with ChatGPT/Claude runs.

In [ ]:
# Install the runner (from this repository) and the GPU libraries
!pip -q install openai transformers accelerate bitsandbytes
!git clone -q https://github.com/putssander/sieve.git 2>/dev/null || true
%cd sieve

In [ ]:
from google.colab import files
up = files.upload()          # choose job-....json
JOB = next(iter(up))
print(JOB)

In [ ]:
# Qwen3-8B fits a T4 in 4-bit. Larger GPUs: drop --4bit, or try Qwen/Qwen3-14B.
MODEL = "Qwen/Qwen3-8B"
!python -m sieve run "$JOB" --backend hf --model $MODEL --4bit

In [ ]:
import glob
files.download(glob.glob('*.results.json')[0])

### Optional: connect the web app live to this GPU
Serve the model with an OpenAI-compatible API (vLLM) and open a temporary tunnel. Paste the printed URL + `/v1` as a *Local / Colab server* in **Models & keys**. The tunnel URL is public while it runs: stop the cell when you are done.

*Not tested yet; the job-file route above is the robust one.*

In [ ]:
# !pip -q install vllm
# !nohup python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-8B --max-model-len 16384 --port 8000 > vllm.log 2>&1 &
# !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
# !./cloudflared tunnel --url http://localhost:8000